In [5]:
import os
import re
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy.stats import norm
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from util import seed_everything

seed_everything(0)

# --- 1. STATISTICAL HELPERS ---

def delong_roc_test(ground_truth, predictions_one, predictions_two):
    """
    Statistically compares two AUCs. Returns (auc1, auc2, p_value).
    """
    auc1 = roc_auc_score(ground_truth, predictions_one)
    auc2 = roc_auc_score(ground_truth, predictions_two)
    pos_mask, neg_mask = (ground_truth == 1), (ground_truth == 0)
    m, n = np.sum(pos_mask), np.sum(neg_mask)
    def get_components(preds):
        pos_preds, neg_preds = preds[pos_mask], preds[neg_mask]
        v10 = np.array([(np.sum(neg_preds < p) + 0.5 * np.sum(neg_preds == p)) / n for p in pos_preds])
        v01 = np.array([(np.sum(pos_preds > p) + 0.5 * np.sum(pos_preds == p)) / m for p in neg_preds])
        return v10, v01
    v10_1, v01_1 = get_components(predictions_one)
    v10_2, v01_2 = get_components(predictions_two)
    var1 = np.var(v10_1, ddof=1)/m + np.var(v01_1, ddof=1)/n
    var2 = np.var(v10_2, ddof=1)/m + np.var(v01_2, ddof=1)/n
    covariance = np.cov(v10_1, v10_2)[0,1]/m + np.cov(v01_1, v01_2)[0,1]/n
    se = np.sqrt(max(1e-10, var1 + var2 - 2 * covariance))
    z = (auc1 - auc2) / se
    return auc1, auc2, 2 * (1 - norm.cdf(np.abs(z)))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# --- 2. MODELS ---

class SpectralViT(nn.Module):
    def __init__(self, n_inputs, n_heads=2, embed_dim=16, n_layers=4):
        super().__init__()
        ranks = torch.arange(1, n_inputs + 1, dtype=torch.float32)
        self.rank_weights = nn.Parameter(1.0 / ranks)
        self.input_proj = nn.Linear(1, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(n_inputs + 1, 1, embed_dim) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim * 2, dropout=0.1)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.mlp_head = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 1))

    def forward(self, x):
        b = x.shape[0]
        x = x * self.rank_weights
        x = self.input_proj(x.unsqueeze(-1)).transpose(0, 1)
        cls_tokens = self.cls_token.expand(1, b, -1)
        x = torch.cat((cls_tokens, x), dim=0)
        x = x + self.pos_embed
        x = self.transformer(x)
        return self.mlp_head(x[0]).squeeze(-1)

class SpatialViT(nn.Module):
    def __init__(self, vol_size=96, patch_size=12, embed_dim=128, n_heads=4, n_layers=2):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (vol_size // patch_size) ** 3
        self.proj = nn.Linear(patch_size ** 3, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(self.n_patches + 1, 1, embed_dim) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim*2, dropout=0.2)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.mlp_head = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 1))

    def forward(self, x):
        b = x.shape[0]
        p = self.patch_size
        x = x.unfold(2, p, p).unfold(3, p, p).unfold(4, p, p).contiguous().view(b, self.n_patches, -1)
        x = self.proj(x).transpose(0, 1)
        cls_tokens = self.cls_token.expand(1, b, -1)
        x = torch.cat((cls_tokens, x), dim=0)
        x = x + self.pos_embed
        x = self.transformer(x)
        return self.mlp_head(x[0]).squeeze(-1)

# --- 3. DATA LOADING ---

def load_ixi_3d(data_dir, csv_path, crop_size=96):
    df = pd.read_csv(csv_path)
    id_col = [c for c in df.columns if 'ID' in c.upper()][0]
    sex_col = [c for c in df.columns if 'SEX' in c.upper()][0]
    df[id_col] = df[id_col].astype(int)
    sex_lookup = dict(zip(df[id_col], df[sex_col].map({1: 0, 2: 1})))
    files = sorted([f for f in os.listdir(data_dir) if f.endswith('.nii.gz')])
    vols, labels = [], []
    for f in tqdm(files):
        match = re.search(r'(\d+)', f)
        if match and int(match.group(1)) in sex_lookup:
            subj_id = int(match.group(1))
            img = nib.load(os.path.join(data_dir, f)).get_fdata()
            pad_widths = [(max(0, (crop_size - s)//2), max(0, (crop_size - s + 1)//2)) for s in img.shape]
            img = np.pad(img, pad_widths, mode='constant')
            c = np.array(img.shape) // 2
            r = crop_size // 2
            crop = img[c[0]-r:c[0]+r, c[1]-r:c[1]+r, c[2]-r:c[2]+r]
            if crop.shape == (crop_size, crop_size, crop_size):
                crop = (crop - np.mean(crop)) / (np.std(crop) + 1e-8)
                vols.append(crop.astype(np.float32)); labels.append(sex_lookup[subj_id])
    return np.array(vols), np.array(labels).astype(np.float32)

# --- 4. TRAINING LOOP WITH OOF COLLECTION ---

MNI_DIR, CSV_PATH = './data/IXI_extracted/', './data/IXI_extracted/IXI.csv'
VOL_SIZE, N_COMP, EPOCHS, BATCH_SIZE = 96, 128, 100, 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X, Y = load_ixi_3d(MNI_DIR, CSV_PATH, crop_size=VOL_SIZE)
X_flat = X.reshape(len(X), -1)

# Parameter Counts
m_spec_tmp = SpectralViT(n_inputs=N_COMP, embed_dim=16)
m_spat_tmp = SpatialViT(vol_size=VOL_SIZE)
params_spec = count_parameters(m_spec_tmp)
params_spat = count_parameters(m_spat_tmp)

# To store predictions across all folds
oof_y_true = []
oof_probs_spec = []
oof_probs_spat = []

kf = KFold(n_splits=5, shuffle=True, random_state=0)

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    pca = PCA(n_components=N_COMP, whiten=True).fit(X_flat[train_idx])
    tr_pca = torch.from_numpy(pca.transform(X_flat[train_idx])).float()
    tr_vol = torch.from_numpy(X[train_idx]).unsqueeze(1).float()
    tr_y = torch.from_numpy(Y[train_idx]).float()
    
    ts_pca = torch.from_numpy(pca.transform(X_flat[test_idx])).float().to(device)
    ts_vol = torch.from_numpy(X[test_idx]).unsqueeze(1).float().to(device)
    ts_y = torch.from_numpy(Y[test_idx]).float().to(device)

    train_loader = DataLoader(TensorDataset(tr_pca, tr_vol, tr_y), batch_size=BATCH_SIZE, shuffle=True)
    m_spec = SpectralViT(n_inputs=N_COMP, embed_dim=16).to(device)
    m_spat = SpatialViT(vol_size=VOL_SIZE).to(device)
    opt_spec = optim.AdamW(m_spec.parameters(), lr=1e-4); opt_spat = optim.AdamW(m_spat.parameters(), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    print(f"\n--- Starting Fold {fold+1} ---")
    for epoch in range(1, EPOCHS + 1):
        m_spec.train(); m_spat.train()
        for b_pca, b_vol, b_y in train_loader:
            b_pca, b_vol, b_y = b_pca.to(device), b_vol.to(device), b_y.to(device)
            opt_spec.zero_grad(); criterion(m_spec(b_pca), b_y).backward(); opt_spec.step()
            opt_spat.zero_grad(); criterion(m_spat(b_vol), b_y).backward(); opt_spat.step()
        
        if epoch % 10 == 0:
                    with torch.no_grad():
                        m_spec.eval(); m_spat.eval()
                        
                        # Get raw probabilities for AUC
                        prob_spec = torch.sigmoid(m_spec(ts_pca))
                        prob_spat = torch.sigmoid(m_spat(ts_vol))
                        
                        # Calculate AUC
                        auc_spec = roc_auc_score(ts_y.cpu().numpy(), prob_spec.cpu().numpy())
                        auc_spat = roc_auc_score(ts_y.cpu().numpy(), prob_spat.cpu().numpy())
                        
                        # Calculate Accuracy
                        acc_spec = ((prob_spec > 0.5).float() == ts_y).float().mean()
                        acc_spat = ((prob_spat > 0.5).float() == ts_y).float().mean()
                        
                        print(f"Fold {fold+1} | Epoch {epoch:3d} | "
                            f"Spec AUC: {auc_spec:.4f} (Acc: {acc_spec:.2%}) | "
                            f"Spat AUC: {auc_spat:.4f} (Acc: {acc_spat:.2%})")

    # Collect Final Fold Predictions
    with torch.no_grad():
        m_spec.eval(); m_spat.eval()
        oof_y_true.append(ts_y.cpu().numpy())
        oof_probs_spec.append(torch.sigmoid(m_spec(ts_pca)).cpu().numpy())
        oof_probs_spat.append(torch.sigmoid(m_spat(ts_vol)).cpu().numpy())

# --- 5. FINAL STATISTICAL SUMMARY ---

y_final = np.concatenate(oof_y_true)
p_spec_final = np.concatenate(oof_probs_spec)
p_spat_final = np.concatenate(oof_probs_spat)

auc_spec, auc_spat, p_val = delong_roc_test(y_final, p_spec_final, p_spat_final)
acc_spec = accuracy_score(y_final, p_spec_final > 0.5)
acc_spat = accuracy_score(y_final, p_spat_final > 0.5)

print("\n" + "="*50)
print("FINAL CROSS-VALIDATION RESULTS")
print("="*50)
print(f"Spectral ViT Mean AUC: {auc_spec:.4f}")
print(f"Spatial ViT  Mean AUC: {auc_spat:.4f}")
print(f"DeLong Test P-Value:   {p_val:.4f}")
print("-" * 50)
print(f"Spectral ViT Accuracy: {acc_spec:.2%}")
print(f"Spatial ViT  Accuracy: {acc_spat:.2%}")
print("="*50)

100%|██████████| 581/581 [01:42<00:00,  5.69it/s]



--- Starting Fold 1 ---
Fold 1 | Epoch  10 | Spec AUC: 0.5704 (Acc: 60.53%) | Spat AUC: 0.6795 (Acc: 57.02%)
Fold 1 | Epoch  20 | Spec AUC: 0.5552 (Acc: 60.53%) | Spat AUC: 0.7246 (Acc: 58.77%)
Fold 1 | Epoch  30 | Spec AUC: 0.7224 (Acc: 64.04%) | Spat AUC: 0.6902 (Acc: 56.14%)
Fold 1 | Epoch  40 | Spec AUC: 0.7981 (Acc: 71.93%) | Spat AUC: 0.6770 (Acc: 59.65%)
Fold 1 | Epoch  50 | Spec AUC: 0.8145 (Acc: 74.56%) | Spat AUC: 0.6828 (Acc: 63.16%)
Fold 1 | Epoch  60 | Spec AUC: 0.8461 (Acc: 72.81%) | Spat AUC: 0.6763 (Acc: 63.16%)
Fold 1 | Epoch  70 | Spec AUC: 0.8651 (Acc: 76.32%) | Spat AUC: 0.6802 (Acc: 63.16%)
Fold 1 | Epoch  80 | Spec AUC: 0.8776 (Acc: 71.05%) | Spat AUC: 0.6818 (Acc: 63.16%)
Fold 1 | Epoch  90 | Spec AUC: 0.8860 (Acc: 77.19%) | Spat AUC: 0.6831 (Acc: 64.04%)
Fold 1 | Epoch 100 | Spec AUC: 0.8908 (Acc: 77.19%) | Spat AUC: 0.6839 (Acc: 64.04%)

--- Starting Fold 2 ---
Fold 2 | Epoch  10 | Spec AUC: 0.5643 (Acc: 57.52%) | Spat AUC: 0.7821 (Acc: 68.14%)
Fold 2 | Epoch 

In [6]:
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar

# 1. Aggregate OOF predictions
y_final = np.concatenate(oof_y_true)
p_spec_final = np.concatenate(oof_probs_spec)
p_spat_final = np.concatenate(oof_probs_spat)
preds_spec = (p_spec_final > 0.5).astype(int)
preds_spat = (p_spat_final > 0.5).astype(int)

# 2. AUC Significance (DeLong)
_, _, p_auc = delong_roc_test(y_final, p_spec_final, p_spat_final)

# 3. Accuracy Significance (McNemar)
# We look at where models disagree
both_correct = np.sum((preds_spec == y_final) & (preds_spat == y_final))
spec_correct_spat_wrong = np.sum((preds_spec == y_final) & (preds_spat != y_final))
spat_correct_spec_wrong = np.sum((preds_spat == y_final) & (preds_spec != y_final))
both_wrong = np.sum((preds_spec != y_final) & (preds_spat != y_final))

table = [[both_correct, spec_correct_spat_wrong],
         [spat_correct_spec_wrong, both_wrong]]

p_acc = mcnemar(table, exact=True).pvalue

# 4. Results
print("\n" + "="*95)
print(f"{'Metric':<25} | {'Spectral ViT':<15} | {'Spatial ViT':<15} | {'P-Value'}")
print("-" * 95)
print(f"{'Mean AUC':<25} | {roc_auc_score(y_final, p_spec_final):<15.4f} | {roc_auc_score(y_final, p_spat_final):<15.4f} | {p_auc:.4f}")
print(f"{'Mean Accuracy':<25} | {accuracy_score(y_final, preds_spec):<15.2%} | {accuracy_score(y_final, preds_spat):<15.2%} | {p_acc:.4f}")
print(f"{'Parameters':<25} | {params_spec:<15,} | {params_spat:<15,} | --")
print("="*95)


Metric                    | Spectral ViT    | Spatial ViT     | P-Value
-----------------------------------------------------------------------------------------------
Mean AUC                  | 0.8302          | 0.6969          | 0.0000
Mean Accuracy             | 75.27%          | 66.43%          | 0.0004
Parameters                | 11,185          | 552,449         | --
